TEST A

In [ ]:
"""
Standalone version: planning engine + Test A (warehouse robot) in one file.
Paste this whole thing into a single Colab cell and run it - no imports
of local files needed.
"""

from collections import deque
from dataclasses import dataclass, field
from typing import FrozenSet, Iterable, List, Optional, Tuple

Proposition = str
State = FrozenSet[Proposition]


@dataclass(frozen=True)
class Action:
    name: str
    pos_preconditions: FrozenSet[Proposition] = field(default_factory=frozenset)
    neg_preconditions: FrozenSet[Proposition] = field(default_factory=frozenset)
    pos_effects: FrozenSet[Proposition] = field(default_factory=frozenset)
    neg_effects: FrozenSet[Proposition] = field(default_factory=frozenset)

    def is_applicable(self, state: State) -> bool:
        return self.pos_preconditions <= state and self.neg_preconditions.isdisjoint(state)

    def apply(self, state: State) -> State:
        if not self.is_applicable(state):
            raise ValueError(f"Action '{self.name}' is not applicable in state {sorted(state)}")
        return (state - self.neg_effects) | self.pos_effects


def make_action(name, pos_pre=(), neg_pre=(), pos_eff=(), neg_eff=()):
    return Action(
        name=name,
        pos_preconditions=frozenset(pos_pre),
        neg_preconditions=frozenset(neg_pre),
        pos_effects=frozenset(pos_eff),
        neg_effects=frozenset(neg_eff),
    )


def goal_satisfied(state, goal_pos, goal_neg):
    return goal_pos <= state and goal_neg.isdisjoint(state)


def bfs_plan(initial_state, actions, goal_pos, goal_neg=()):
    start = frozenset(initial_state)
    goal_pos = frozenset(goal_pos)
    goal_neg = frozenset(goal_neg)

    if goal_satisfied(start, goal_pos, goal_neg):
        return []

    frontier = deque([(start, [])])
    visited = {start}

    while frontier:
        state, path = frontier.popleft()
        for action in actions:
            if not action.is_applicable(state):
                continue
            new_state = action.apply(state)
            if new_state in visited:
                continue
            new_path = path + [(action, new_state)]
            if goal_satisfied(new_state, goal_pos, goal_neg):
                return new_path
            visited.add(new_state)
            frontier.append((new_state, new_path))

    return None


def print_plan(initial_state, plan):
    print("Initial state:")
    print(f"  {sorted(initial_state)}\n")

    if plan is None:
        print("No plan exists: the goal is unreachable from the initial state "
              "using the given actions.")
        return

    if not plan:
        print("Goal already satisfied in the initial state. No actions needed.")
        return

    print(f"Plan found ({len(plan)} action(s)):\n")
    for step, (action, state_after) in enumerate(plan, start=1):
        print(f"Step {step}: {action.name}")
        print(f"  State after: {sorted(state_after)}\n")


# ===========================================================================
# Test A: Warehouse robot problem
# ===========================================================================

initial_state = {
    "robot_at_A",
    "package_at_A",
}

actions = [
    make_action("Move(A,B)", pos_pre=["robot_at_A"], pos_eff=["robot_at_B"], neg_eff=["robot_at_A"]),
    make_action("Move(B,A)", pos_pre=["robot_at_B"], pos_eff=["robot_at_A"], neg_eff=["robot_at_B"]),
    make_action("Move(B,C)", pos_pre=["robot_at_B"], pos_eff=["robot_at_C"], neg_eff=["robot_at_B"]),
    make_action("Move(C,B)", pos_pre=["robot_at_C"], pos_eff=["robot_at_B"], neg_eff=["robot_at_C"]),

    make_action("PickUp(Package,A)", pos_pre=["robot_at_A", "package_at_A"],
                pos_eff=["holding_package"], neg_eff=["package_at_A"]),
    make_action("PickUp(Package,B)", pos_pre=["robot_at_B", "package_at_B"],
                pos_eff=["holding_package"], neg_eff=["package_at_B"]),
    make_action("PickUp(Package,C)", pos_pre=["robot_at_C", "package_at_C"],
                pos_eff=["holding_package"], neg_eff=["package_at_C"]),

    make_action("Drop(Package,A)", pos_pre=["robot_at_A", "holding_package"],
                pos_eff=["package_at_A"], neg_eff=["holding_package"]),
    make_action("Drop(Package,B)", pos_pre=["robot_at_B", "holding_package"],
                pos_eff=["package_at_B"], neg_eff=["holding_package"]),
    make_action("Drop(Package,C)", pos_pre=["robot_at_C", "holding_package"],
                pos_eff=["package_at_C"], neg_eff=["holding_package"]),
]

goal_pos = {"package_at_C"}
goal_neg = set()

plan = bfs_plan(initial_state, actions, goal_pos, goal_neg)
print_plan(initial_state, plan)

Initial state:
  ['package_at_A', 'robot_at_A']

Plan found (4 action(s)):

Step 1: PickUp(Package,A)
  State after: ['holding_package', 'robot_at_A']

Step 2: Move(A,B)
  State after: ['holding_package', 'robot_at_B']

Step 3: Move(B,C)
  State after: ['holding_package', 'robot_at_C']

Step 4: Drop(Package,C)
  State after: ['package_at_C', 'robot_at_C']



TEST B

In [ ]:
"""
Standalone version: planning engine + Test A (warehouse robot) in one file.
Paste this whole thing into a single Colab cell and run it - no imports
of local files needed.
"""

from collections import deque
from dataclasses import dataclass, field
from typing import FrozenSet, Iterable, List, Optional, Tuple

Proposition = str
State = FrozenSet[Proposition]


@dataclass(frozen=True)
class Action:
    name: str
    pos_preconditions: FrozenSet[Proposition] = field(default_factory=frozenset)
    neg_preconditions: FrozenSet[Proposition] = field(default_factory=frozenset)
    pos_effects: FrozenSet[Proposition] = field(default_factory=frozenset)
    neg_effects: FrozenSet[Proposition] = field(default_factory=frozenset)

    def is_applicable(self, state: State) -> bool:
        return self.pos_preconditions <= state and self.neg_preconditions.isdisjoint(state)

    def apply(self, state: State) -> State:
        if not self.is_applicable(state):
            raise ValueError(f"Action '{self.name}' is not applicable in state {sorted(state)}")
        return (state - self.neg_effects) | self.pos_effects


def make_action(name, pos_pre=(), neg_pre=(), pos_eff=(), neg_eff=()):
    return Action(
        name=name,
        pos_preconditions=frozenset(pos_pre),
        neg_preconditions=frozenset(neg_pre),
        pos_effects=frozenset(pos_eff),
        neg_effects=frozenset(neg_eff),
    )


def goal_satisfied(state, goal_pos, goal_neg):
    return goal_pos <= state and goal_neg.isdisjoint(state)


def bfs_plan(initial_state, actions, goal_pos, goal_neg=()):
    start = frozenset(initial_state)
    goal_pos = frozenset(goal_pos)
    goal_neg = frozenset(goal_neg)

    if goal_satisfied(start, goal_pos, goal_neg):
        return []

    frontier = deque([(start, [])])
    visited = {start}

    while frontier:
        state, path = frontier.popleft()
        for action in actions:
            if not action.is_applicable(state):
                continue
            new_state = action.apply(state)
            if new_state in visited:
                continue
            new_path = path + [(action, new_state)]
            if goal_satisfied(new_state, goal_pos, goal_neg):
                return new_path
            visited.add(new_state)
            frontier.append((new_state, new_path))

    return None


def print_plan(initial_state, plan):
    print("Initial state:")
    print(f"  {sorted(initial_state)}\n")

    if plan is None:
        print("No plan exists: the goal is unreachable from the initial state "
              "using the given actions.")
        return

    if not plan:
        print("Goal already satisfied in the initial state. No actions needed.")
        return

    print(f"Plan found ({len(plan)} action(s)):\n")
    for step, (action, state_after) in enumerate(plan, start=1):
        print(f"Step {step}: {action.name}")
        print(f"  State after: {sorted(state_after)}\n")


# ===========================================================================
# Test A: Warehouse robot problem
# ===========================================================================

initial_state = {
    "robot_at_A",
    "package_at_A",
}

actions = [
    make_action("Move(A,B)", pos_pre=["robot_at_A"], pos_eff=["robot_at_B"], neg_eff=["robot_at_A"]),
    make_action("Move(B,A)", pos_pre=["robot_at_B"], pos_eff=["robot_at_A"], neg_eff=["robot_at_B"]),
    make_action("Move(B,C)", pos_pre=["robot_at_B"], pos_eff=["robot_at_C"], neg_eff=["robot_at_B"]),
    make_action("Move(C,B)", pos_pre=["robot_at_C"], pos_eff=["robot_at_B"], neg_eff=["robot_at_C"]),



    make_action("Drop(Package,A)", pos_pre=["robot_at_A", "holding_package"],
                pos_eff=["package_at_A"], neg_eff=["holding_package"]),
    make_action("Drop(Package,B)", pos_pre=["robot_at_B", "holding_package"],
                pos_eff=["package_at_B"], neg_eff=["holding_package"]),
    make_action("Drop(Package,C)", pos_pre=["robot_at_C", "holding_package"],
                pos_eff=["package_at_C"], neg_eff=["holding_package"]),
]

goal_pos = {"package_at_C"}
goal_neg = set()

plan = bfs_plan(initial_state, actions, goal_pos, goal_neg)
print_plan(initial_state, plan)

Initial state:
  ['package_at_A', 'robot_at_A']

No plan exists: the goal is unreachable from the initial state using the given actions.


TEST C

In [ ]:
"""
Standalone version: planning engine + Test A (warehouse robot) in one file.
Paste this whole thing into a single Colab cell and run it - no imports
of local files needed.
"""

from collections import deque
from dataclasses import dataclass, field
from typing import FrozenSet, Iterable, List, Optional, Tuple

Proposition = str
State = FrozenSet[Proposition]


@dataclass(frozen=True)
class Action:
    name: str
    pos_preconditions: FrozenSet[Proposition] = field(default_factory=frozenset)
    neg_preconditions: FrozenSet[Proposition] = field(default_factory=frozenset)
    pos_effects: FrozenSet[Proposition] = field(default_factory=frozenset)
    neg_effects: FrozenSet[Proposition] = field(default_factory=frozenset)

    def is_applicable(self, state: State) -> bool:
        return self.pos_preconditions <= state and self.neg_preconditions.isdisjoint(state)

    def apply(self, state: State) -> State:
        if not self.is_applicable(state):
            raise ValueError(f"Action '{self.name}' is not applicable in state {sorted(state)}")
        return (state - self.neg_effects) | self.pos_effects


def make_action(name, pos_pre=(), neg_pre=(), pos_eff=(), neg_eff=()):
    return Action(
        name=name,
        pos_preconditions=frozenset(pos_pre),
        neg_preconditions=frozenset(neg_pre),
        pos_effects=frozenset(pos_eff),
        neg_effects=frozenset(neg_eff),
    )


def goal_satisfied(state, goal_pos, goal_neg):
    return goal_pos <= state and goal_neg.isdisjoint(state)


def bfs_plan(initial_state, actions, goal_pos, goal_neg=()):
    start = frozenset(initial_state)
    goal_pos = frozenset(goal_pos)
    goal_neg = frozenset(goal_neg)

    if goal_satisfied(start, goal_pos, goal_neg):
        return []

    frontier = deque([(start, [])])
    visited = {start}

    while frontier:
        state, path = frontier.popleft()
        for action in actions:
            if not action.is_applicable(state):
                continue
            new_state = action.apply(state)
            if new_state in visited:
                continue
            new_path = path + [(action, new_state)]
            if goal_satisfied(new_state, goal_pos, goal_neg):
                return new_path
            visited.add(new_state)
            frontier.append((new_state, new_path))

    return None


def print_plan(initial_state, plan):
    print("Initial state:")
    print(f"  {sorted(initial_state)}\n")

    if plan is None:
        print("No plan exists: the goal is unreachable from the initial state "
              "using the given actions.")
        return

    if not plan:
        print("Goal already satisfied in the initial state. No actions needed.")
        return

    print(f"Plan found ({len(plan)} action(s)):\n")
    for step, (action, state_after) in enumerate(plan, start=1):
        print(f"Step {step}: {action.name}")
        print(f"  State after: {sorted(state_after)}\n")


# ===========================================================================
# Test A: Warehouse robot problem
# ===========================================================================

initial_state = {
    "robot_at_A",
    "package_at_A",
}

actions = [
    make_action("Move(A,B)", pos_pre=["robot_at_A"], pos_eff=["robot_at_B"], neg_eff=["robot_at_A"]),
    make_action("Move(B,A)", pos_pre=["robot_at_B"], pos_eff=["robot_at_A"], neg_eff=["robot_at_B"]),
    make_action("Move(B,C)", pos_pre=["robot_at_B"], pos_eff=["robot_at_C"], neg_eff=["robot_at_B"]),
    make_action("Move(C,B)", pos_pre=["robot_at_C"], pos_eff=["robot_at_B"], neg_eff=["robot_at_C"]),

    make_action("PickUp(Package,A)", pos_pre=["robot_at_A", "package_at_A"],
                pos_eff=["holding_package"], neg_eff=["package_at_A"]),
    make_action("PickUp(Package,B)", pos_pre=["robot_at_B", "package_at_B"],
                pos_eff=["holding_package"], neg_eff=["package_at_B"]),
    make_action("PickUp(Package,C)", pos_pre=["robot_at_C", "package_at_C"],
                pos_eff=["holding_package"], neg_eff=["package_at_C"]),

    make_action("Drop(Package,A)", pos_pre=["robot_at_A", "holding_package"],
                pos_eff=["package_at_A"], neg_eff=["holding_package"]),
    make_action("Drop(Package,B)", pos_pre=["robot_at_B", "holding_package"],
                pos_eff=["package_at_B"], neg_eff=["holding_package"]),
    make_action("Drop(Package,C)", pos_pre=["robot_at_C", "holding_package"],
                pos_eff=["package_at_C"], neg_eff=["holding_package"]),
]

goal_pos = {"robot_at_C"}
goal_neg = set()

plan = bfs_plan(initial_state, actions, goal_pos, goal_neg)
print_plan(initial_state, plan)

Initial state:
  ['package_at_A', 'robot_at_A']

Plan found (2 action(s)):

Step 1: Move(A,B)
  State after: ['package_at_A', 'robot_at_B']

Step 2: Move(B,C)
  State after: ['package_at_A', 'robot_at_C']

